# Irish Property Prices - Data Exploration

Notebook 1 of 4. This notebook inspects the two raw datasets, documents their
structure and quality issues, and establishes the decisions that shape the rest
of the project. No cleaning is performed here - that happens in notebook 02.

**Data sources**

- **Property Price Register (PPR)** - every residential property sale in Ireland
  since 2010. These are actual sale prices filed for stamp duty purposes, not
  asking prices. Downloaded from propertypriceregister.ie
- **SEAI National BER Research Tool** - anonymised domestic Building Energy
  Rating records. Provides property characteristics such as floor area, dwelling
  type, year of construction and energy rating. Downloaded from ndber.seai.ie

Both files are downloaded manually and placed in `data/raw/`. Neither source
offers a stable direct-download URL suitable for automation.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")

print("Files in raw data directory:")
for f in sorted(RAW.iterdir()):
    size_mb = f.stat().st_size / 1_048_576
    print(f"  {f.name}  ({size_mb:,.1f} MB)")

Files in raw data directory:
  BERPublicsearch.txt  (1,465.5 MB)
  ppr_raw.csv  (94.2 MB)


## 2. Property Price Register

The PPR export is latin-1 encoded rather than UTF-8. Reading it as UTF-8 either
fails outright or mangles Irish-language characters and the euro symbol in the
price column.

In [2]:
PPR_FILE = "ppr_raw.csv"

ppr = pd.read_csv(RAW / PPR_FILE, encoding="latin-1")

print(f"Shape: {ppr.shape[0]:,} rows x {ppr.shape[1]} columns")
print(f"\nColumns:")
for col in ppr.columns:
    print(f"  {col}")

Shape: 799,067 rows x 9 columns

Columns:
  Date of Sale (dd/mm/yyyy)
  Address
  County
  Eircode
  Price ()
  Not Full Market Price
  VAT Exclusive
  Description of Property
  Property Size Description


C:\Users\heffo\AppData\Local\Temp\ipykernel_8720\1431381199.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  ppr = pd.read_csv(RAW / PPR_FILE, encoding="latin-1")


In [3]:
display(ppr.head(10))

,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description
0,1/01/2010,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN," 343,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
1,3/01/2010,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN," 185,000.00",No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...
2,4/01/2010,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN," 438,500.00",No,No,Second-Hand Dwelling house /Apartment,NaN
3,4/01/2010,"1 The Haven, Mornington",Meath,NaN," 400,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
4,4/01/2010,"11 Melville Heights, Kilkenny",Kilkenny,NaN," 160,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
5,4/01/2010,"12 Sallymount Avenue, Ranelagh",Dublin,NaN," 425,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
6,4/01/2010,"13 Oakleigh Wood, Dooradoyle, Limerick",Limerick,NaN," 172,500.00",No,No,Second-Hand Dwelling house /Apartment,NaN
7,4/01/2010,"13 The Drive, Chapelstown Gate, Tullow Road",Carlow,NaN," 177,500.00",No,No,Second-Hand Dwelling house /Apartment,NaN
8,4/01/2010,"15 Carriglawn, Waterpark, Carrigaline",Cork,NaN," 180,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
9,4/01/2010,"15a Moore Bay, Kilkee",Clare,NaN," 126,500.00",No,No,Second-Hand Dwelling house /Apartment,NaN


In [4]:
print("Missing values per column:")
missing = pd.DataFrame({
    "nulls": ppr.isnull().sum(),
    "pct": (ppr.isnull().sum() / len(ppr) * 100).round(1)
})
print(missing.to_string())

Missing values per column:
                            nulls   pct
Date of Sale (dd/mm/yyyy)       0   0.0
Address                         0   0.0
County                          0   0.0
Eircode                    555111  69.5
Price ()                       0   0.0
Not Full Market Price           0   0.0
VAT Exclusive                   0   0.0
Description of Property         0   0.0
Property Size Description  746222  93.4


### 2.1 Column-by-column inspection

Categorical columns are shown as value counts, high-cardinality columns as a
sample of values.

In [5]:
for col in ppr.columns:
    n_unique = ppr[col].nunique()
    print(f"\n{'=' * 60}")
    print(f"{col}")
    print(f"  dtype: {ppr[col].dtype}  |  unique: {n_unique:,}  |  nulls: {ppr[col].isnull().sum():,}")
    if n_unique <= 10:
        print(ppr[col].value_counts(dropna=False).to_string())
    else:
        print(f"  sample: {ppr[col].dropna().head(5).tolist()}")


Date of Sale (dd/mm/yyyy)
  dtype: object  |  unique: 5,708  |  nulls: 0
  sample: ['1/01/2010', '3/01/2010', '4/01/2010', '4/01/2010', '4/01/2010']

Address
  dtype: object  |  unique: 704,485  |  nulls: 0
  sample: ['5 Braemor Drive, Churchtown, Co.Dublin', '134 Ashewood Walk, Summerhill Lane, Portlaoise', '1 Meadow Avenue, Dundrum, Dublin 14', '1 The Haven, Mornington', '11 Melville Heights, Kilkenny']

County
  dtype: object  |  unique: 26  |  nulls: 0
  sample: ['Dublin', 'Laois', 'Dublin', 'Meath', 'Kilkenny']

Eircode
  dtype: object  |  unique: 231,794  |  nulls: 555,111
  sample: ['D16R293', 'P43PW95', 'T12N6X0', 'T23A7X8', 'D11TX45']

Price ()
  dtype: object  |  unique: 44,054  |  nulls: 0
  sample: ['\x80 343,000.00', '\x80 185,000.00', '\x80 438,500.00', '\x80 400,000.00', '\x80 160,000.00']

Not Full Market Price
  dtype: object  |  unique: 2  |  nulls: 0
Not Full Market Price
No     758346
Yes     40721

VAT Exclusive
  dtype: object  |  unique: 2  |  nulls: 0
VAT Excl

### 2.2 Data quality issues

Four issues are visible from the inspection above and will need handling in the
cleaning stage.

**Price is stored as text.** Values appear as `'\x80 343,000.00'` - the euro
symbol mangled by the latin-1 encoding, plus comma thousand separators. The
column needs the symbol stripped, commas removed and conversion to float.

**New builds are listed excluding VAT.** The `VAT Exclusive` flag identifies
these. Irish VAT on new residential property is 13.5%, so these prices need
multiplying by 1.135 to be comparable with second-hand sales. Failing to do this
systematically understates new build values.

**Some sales are not arms-length.** The `Not Full Market Price` flag marks
transfers between related parties, family transactions and similar. These are
not market prices and should be excluded from any price model.

**Irish-language duplicate categories.** `Description of Property` and
`Property Size Description` both contain Irish-language equivalents of the
English categories, plus mojibake variants where the encoding has failed. These
represent a small number of rows but need mapping onto the English categories
rather than being treated as separate types.

In [6]:
print("Price column - raw sample:")
print(ppr["Price (\x80)"].head(5).tolist())

# parse a copy for inspection only - actual cleaning happens in notebook 02
price_check = (ppr["Price (\x80)"]
               .str.replace("\x80", "", regex=False)
               .str.replace(",", "", regex=False)
               .str.strip()
               .astype(float))

print(f"\nParsed price summary:")
print(price_check.describe().apply(lambda v: f"{v:,.0f}").to_string())

Price column - raw sample:
['\x80 343,000.00', '\x80 185,000.00', '\x80 438,500.00', '\x80 400,000.00', '\x80 160,000.00']

Parsed price summary:
count        799,067
mean         319,174
std        1,234,377
min            5,001
25%          145,000
50%          243,500
75%          365,000
max      387,665,198


In [7]:
print("Flag columns:")
print(f"\nNot Full Market Price:")
print(ppr["Not Full Market Price"].value_counts().to_string())
print(f"\nVAT Exclusive:")
print(ppr["VAT Exclusive"].value_counts().to_string())

print(f"\n\nDescription of Property:")
print(ppr["Description of Property"].value_counts(dropna=False).to_string())

print(f"\n\nProperty Size Description:")
print(ppr["Property Size Description"].value_counts(dropna=False).to_string())

Flag columns:

Not Full Market Price:
Not Full Market Price
No     758346
Yes     40721

VAT Exclusive:
VAT Exclusive
No     658693
Yes    140374


Description of Property:
Description of Property
Second-Hand Dwelling house /Apartment    656283
New Dwelling house /Apartment            142735
Teach/Árasán Cónaithe Atháimhe               45
Teach/Árasán Cónaithe Nua                     3
Teach/?ras?n C?naithe Nua                     1


Property Size Description:
Property Size Description
NaN                                                                                746222
greater than or equal to 38 sq metres and less than 125 sq metres                   38106
greater than 125 sq metres                                                           6847
greater than or equal to 125 sq metres                                               4622
less than 38 sq metres                                                               3267
níos mó ná nó cothrom le 38 méadar cearnach agus níos lú n

`Property Size Description` is 93% null and the non-null values form only three
overlapping bands, with an ambiguity between "greater than 125 sq metres" and
"greater than or equal to 125 sq metres". This column is not usable as a size
feature and will be dropped.

### 2.3 Sale dates

In [8]:
ppr["date"] = pd.to_datetime(ppr["Date of Sale (dd/mm/yyyy)"], format="%d/%m/%Y")
ppr["year"] = ppr["date"].dt.year

print(f"Date range: {ppr['date'].min().date()} to {ppr['date'].max().date()}")
print(f"\nSales per year:")
print(ppr["year"].value_counts().sort_index().to_string())

Date range: 2010-01-01 to 2026-07-31

Sales per year:
year
2010    21004
2011    18446
2012    25375
2013    30226
2014    43675
2015    49166
2016    49941
2017    55023
2018    57463
2019    59066
2020    49559
2021    59604
2022    62756
2023    63383
2024    61564
2025    62073
2026    30743


### 2.4 Eircode coverage

Eircode is the field that determines how geographically granular this project
can be. County alone is too coarse to be useful for price modelling, since
Dublin is a single county containing enormously varied submarkets.

Eircodes launched in Ireland in 2015, but coverage in the PPR depends on filing
practice rather than availability.

In [9]:
coverage = ppr.groupby("year").agg(
    sales=("Eircode", "size"),
    with_eircode=("Eircode", "count")
)
coverage["pct"] = (coverage["with_eircode"] / coverage["sales"] * 100).round(1)
print(coverage.to_string())

      sales  with_eircode   pct
year                           
2010  21004             1   0.0
2011  18446             2   0.0
2012  25375             0   0.0
2013  30226            26   0.1
2014  43675            60   0.1
2015  49166            44   0.1
2016  49941            81   0.2
2017  55023           112   0.2
2018  57463           117   0.2
2019  59066           194   0.3
2020  49559           385   0.8
2021  59604         30637  51.4
2022  62756         48258  76.9
2023  63383         48362  76.3
2024  61564         46291  75.2
2025  62073         46000  74.1
2026  30743         23386  76.1


Coverage is effectively zero through 2020, jumps to 51% in 2021 and settles at
74-77% from 2022 onward. Something changed in PPR filing practice in 2021.

**This establishes the modelling window.** Any analysis relying on Eircode
granularity uses 2021 onward. The full 2010-present series remains usable for
county-level trend analysis.

### 2.5 Eircode format validation

A valid Eircode is a routing key of one letter followed by two digits, 
then a four-character alphanumeric unique identifier - for example D16R293.
Dublin 6W is the single exception in the system, using a three-character routing key. 
The validation pattern accounts for this.

In [10]:
recent = ppr[ppr["year"] >= 2021].copy()

pattern = r"^([A-Za-z]\d{2}|D6W)\s?[A-Za-z0-9]{4}$"
has_eircode = recent["Eircode"].notna()
valid_format = recent["Eircode"].str.match(pattern, na=False)

print(f"Rows 2021 onward:        {len(recent):,}")
print(f"With an Eircode:         {has_eircode.sum():,}")
print(f"Format-valid:            {valid_format.sum():,}")
print(f"Present but malformed:   {(has_eircode & ~valid_format).sum():,}")

bad = recent.loc[has_eircode & ~valid_format, "Eircode"]
if len(bad):
    print(f"\nMost common malformed values:")
    print(bad.value_counts().head(20).to_string())

Rows 2021 onward:        340,123
With an Eircode:         242,934
Format-valid:            242,934
Present but malformed:   0


All 1,558 initially malformed values were Dublin 6W addresses rejected by a pattern that assumed the standard format. With D6W accounted for, 242,934 of 242,934 present Eircodes are format-valid.

In [11]:
# check for placeholder or repeated values among valid-format Eircodes
dupes = recent.loc[valid_format, "Eircode"].value_counts()
print("Most frequently repeated Eircodes:")
print(dupes.head(15).to_string())
print(f"\nEircodes appearing more than 20 times: {(dupes > 20).sum()}")

Most frequently repeated Eircodes:
Eircode
A123456    156
D24W9NN     34
V15KH39     10
A96WV79      8
W23VH52      8
D24V529      7
D09V9R3      7
D07F6K5      7
D03Y519      7
A00AA00      7
D01PK61      6
D02R256      6
D08NPH4      6
A65F4E2      6
A96NP89      6

Eircodes appearing more than 20 times: 2


Two placeholder values are present. A123456 appears 156 times and A00AA00 seven times, both filed against unrelated addresses across the country. These are dummy entries rather than real Eircodes and are excluded during cleaning.

D24W9NN appears 34 times but inspection shows these are genuine sales - 33 units at a single Citywest development filed on the same date, sharing one Eircode as a filing shortcut. The routing key D24 is correct for these and they are retained. The repeated prices to the cent across units do suggest a bulk transaction rather than independent market sales, which is noted for consideration during modelling.

In [12]:
ppr["price"] = (ppr["Price (\x80)"]
                .str.replace("\x80", "", regex=False)
                .str.replace(",", "", regex=False)
                .str.strip()
                .astype(float))

recent = ppr[ppr["year"] >= 2021].copy()

In [13]:
chk = recent[recent["Eircode"] == "D24W9NN"]
print(chk[["date", "Address", "price"]].head(20).to_string())

             date                                                 Address      price
519244 2022-01-12  3 CITYWEST VILLAGE AVENUE, CITYWEST VILLAGE, DUBLIN 24  374443.33
639664 2023-12-11                          1 THE OAKS, BARNOAKS, CITYWEST  319874.19
639674 2023-12-11                         10 THE OAKS, BARNOAKS, CITYWEST  319874.19
639685 2023-12-11                         11 THE OAKS, BARNOAKS, CITYWEST  319874.19
639693 2023-12-11                         12 THE OAKS, BARNOAKS, CITYWEST  304724.78
639701 2023-12-11                         13 THE OAKS, BARNOAKS, CITYWEST  235199.81
639709 2023-12-11                         14 THE OAKS, BARNOAKS, CITYWEST  319874.19
639717 2023-12-11                         15 THE OAKS, BARNOAKS, CITYWEST  319874.19
639722 2023-12-11                         16 THE OAKS, BARNOAKS, CITYWEST  235199.81
639731 2023-12-11                         17 THE OAKS, BARNOAKS, CITYWEST  176209.69
639736 2023-12-11                         18 THE OAKS, BARNOAKS, 

A genuine Eircode identifies a single property, so a small number of repeats is
expected where the same property has sold more than once. Any value repeating
hundreds of times is a placeholder and those rows should be excluded.

### 2.6 Routing keys

The routing key is the first three characters of an Eircode and identifies a
geographic area rather than an individual property - `D16` is Dundrum and
surrounds, `T12` is a part of Cork city. Unlike full Eircodes, routing keys are
freely usable and are the right level of granularity both for modelling and for
a lookup tool.

In [14]:
recent["routing_key"] = recent.loc[valid_format, "Eircode"].str[:3].str.upper()

print(f"Distinct routing keys: {recent['routing_key'].nunique()}")
print(f"\nMost common:")
print(recent["routing_key"].value_counts().head(20).to_string())

Distinct routing keys: 306

Most common:
routing_key
V94    8748
H91    7632
T12    6951
D15    6856
X91    5671
D24    4856
A92    4842
W91    4568
D18    4315
R32    3923
D08    3874
D04    3739
A94    3558
C15    3532
A96    3527
T23    3430
D13    3421
D09    3349
Y35    3343
A91    3342


In [15]:
# do routing keys respect county boundaries?
xref = (recent.dropna(subset=["routing_key"])
        .groupby("routing_key")["County"]
        .nunique()
        .sort_values(ascending=False))

print(f"Routing keys spanning multiple counties: {(xref > 1).sum()} of {len(xref)}")
print(f"\nMost cross-county:")
print(xref.head(15).to_string())

Routing keys spanning multiple counties: 143 of 306

Most cross-county:
routing_key
A12    26
V94    14
W91    14
H91    12
N41    11
N91    11
A96    11
F91    11
C15    10
W23    10
R32     9
A82     9
V93     8
F26     8
F12     8


In [16]:
# inspect the worst offender to determine whether this is real geography
# or a data quality artefact
worst = xref.index[0]
sample = recent[recent["routing_key"] == worst]

print(f"Routing key {worst}: {len(sample):,} sales across {sample['County'].nunique()} counties")
print(f"Distinct full Eircodes: {sample['Eircode'].nunique():,}")
print(f"\nCounty spread:")
print(sample["County"].value_counts().head(10).to_string())
print(f"\nSample rows:")
print(sample[["Address", "County", "Eircode"]].head(10).to_string())

Routing key A12: 158 sales across 26 counties
Distinct full Eircodes: 3

County spread:
County
Cork        22
Dublin      19
Donegal     14
Kildare     11
Kilkenny    10
Galway       9
Wicklow      8
Cavan        8
Limerick     7
Mayo         6

Sample rows:
                                                     Address     County  Eircode
508844            DUPLEX  APARTMENT, 98 DOUGLAS STREET, CORK       Cork  A12AP2D
735845                   MEENACLADDY, DORTAHORK, LETTERKENNY    Donegal  A123456
748774                      RATHOSEY, COOLANEY, COUNTY SLIGO      Sligo  A123456
750453                           CUILMORE, SWINFORD, CO MAYO       Mayo  A123456
751411                   LISSACOPPLE, VIRGINIA, COUNTY CAVAN      Cavan  A123456
751734                       NEWFOREST, KILBEGGAN, WESTMEATH  Westmeath  A123456
751935                         COOLCOULAGHTA, DURRUS, BANTRY       Cork  A123456
754071           33 CHARLEVILLE ROAD, PHIBSBOROUGH, DUBLIN 7     Dublin  A123456
754254      

143 of 306 routing keys initially appear to span multiple counties. Investigation shows this is largely an artefact of the A123456 placeholder: routing key A12 appears across 26 counties but contains only three distinct Eircodes, 156 of its 158 sales being the placeholder value filed against addresses in Donegal, Sligo, Mayo, Cavan, Westmeath, Cork, Dublin and Leitrim.

Once placeholders are excluded, the remaining cross-county spread is genuine geography. Routing key areas follow postal delivery patterns rather than administrative boundaries, so keys serving towns near county borders legitimately appear in both - V94 around Limerick and H91 around Galway being the clearest examples.

### 2.7 Is Eircode coverage geographically even?

If some counties have far lower coverage than others, any model trained on the
Eircode subset will perform unevenly across the country. This needs to be known
and disclosed rather than discovered later.

In [17]:
geo_cov = recent.groupby("County").agg(
    sales=("Eircode", "size"),
    with_eircode=("Eircode", "count")
)
geo_cov["pct"] = (geo_cov["with_eircode"] / geo_cov["sales"] * 100).round(1)
print(geo_cov.sort_values("pct").to_string())

            sales  with_eircode   pct
County                               
Kildare     20447         11483  56.2
Meath       14963          9088  60.7
Wicklow     12712          7749  61.0
Laois        6375          4106  64.4
Louth       10188          6573  64.5
Cork        38280         25866  67.6
Kilkenny     5673          3844  67.8
Donegal      8623          5977  69.3
Offaly       4710          3271  69.4
Wexford     12345          8681  70.3
Sligo        4857          3462  71.3
Westmeath    6558          4673  71.3
Waterford    9673          6908  71.4
Mayo         8174          5861  71.7
Roscommon    4722          3391  71.8
Galway      15421         11165  72.4
Monaghan     2732          1981  72.5
Kerry        8401          6104  72.7
Carlow       3749          2761  73.6
Clare        7396          5440  73.6
Limerick    12129          9028  74.4
Cavan        4719          3528  74.8
Leitrim      2873          2158  75.1
Tipperary    9061          6901  76.2
Longford    

## 3. SEAI BER Research Tool

The BER export is a tab-delimited text file despite the site describing it as a
spreadsheet. It is also latin-1 encoded, contains 252 columns, and pads every
string field with trailing spaces to a fixed width.

The padding matters - left unhandled, `"Detached house"` and
`"Detached house    "` are treated as separate categories and any grouping
silently splits.

In [18]:
BER_FILE = "BERPublicsearch.txt"
ber_path = RAW / BER_FILE

# confirm the delimiter before parsing
with open(ber_path, encoding="latin-1") as f:
    header = f.readline()

print("Delimiter counts in header row:")
for name, ch in [("tab", "\t"), ("pipe", "|"), ("comma", ","), ("semicolon", ";")]:
    print(f"  {name:10s} {header.count(ch)}")

Delimiter counts in header row:
  tab        251
  pipe       0
  comma      0
  semicolon  0


In [19]:
ber = pd.read_csv(
    ber_path,
    sep="\t",
    encoding="latin-1",
    low_memory=False,
    on_bad_lines="skip",   # free-text fields occasionally contain stray tabs
    quoting=3              # csv.QUOTE_NONE - unbalanced quotes otherwise swallow rows
)

# strip the fixed-width padding from every text column
for col in ber.select_dtypes("object").columns:
    ber[col] = ber[col].str.strip()

print(f"Shape: {ber.shape[0]:,} rows x {ber.shape[1]} columns")

Shape: 1,430,031 rows x 252 columns


In [20]:
print("All columns:")
for i, col in enumerate(ber.columns):
    print(f"{i:3d}  {col}")

All columns:
  0  CountyName
  1  DwellingTypeDescr
  2  Year_of_Construction
  3  TypeofRating
  4  EnergyRating
  5  BerRating
  6  GroundFloorArea(sq m)
  7  UValueWall
  8  UValueRoof
  9  UValueFloor
 10  UValueWindow
 11  UvalueDoor
 12  WallArea
 13  RoofArea
 14  FloorArea
 15  WindowArea
 16  DoorArea
 17  NoStoreys
 18  CO2Rating
 19  MainSpaceHeatingFuel
 20  MainWaterHeatingFuel
 21  HSMainSystemEfficiency
 22  MultiDwellingMPRN
 23  TGDLEdition
 24  MPCDERValue
 25  HSEffAdjFactor
 26  HSSupplHeatFraction
 27  HSSupplSystemEff
 28  WHMainSystemEff
 29  WHEffAdjFactor
 30  SupplSHFuel
 31  SupplWHFuel
 32  SHRenewableResources
 33  WHRenewableResources
 34  NoOfChimneys
 35  NoOfOpenFlues
 36  NoOfFansAndVents
 37  NoOfFluelessGasFires
 38  DraftLobby
 39  VentilationMethod
 40  FanPowerManuDeclaredValue
 41  HeatExchangerEff
 42  StructureType
 43  SuspendedWoodenFloor
 44  PercentageDraughtStripped
 45  NoOfSidesSheltered
 46  PermeabilityTest
 47  PermeabilityTestResult


### 3.1 What geography survives anonymisation?

This is the critical question for the BER dataset. The full BER data file
includes the property address and MPRN, both of which are personal data under
GDPR. The public research extract is anonymised, so the question is what
geographic detail remains.

The schema contains an `SA_Code` column - the CSO Small Area code, a census
geography of roughly 80-120 dwellings. If populated this would be extremely
granular. It needs checking.

In [21]:
print(f"SA_Code nulls: {ber['SA_Code'].isnull().sum():,} / {len(ber):,}")
print(f"Distinct Small Areas: {ber['SA_Code'].nunique():,}")

for col in ["prob_smarea_error_0corr", "prob_smarea_error_100corr"]:
    print(f"{col} nulls: {ber[col].isnull().sum():,} / {len(ber):,}")

SA_Code nulls: 1,430,031 / 1,430,031
Distinct Small Areas: 0
prob_smarea_error_0corr nulls: 1,430,031 / 1,430,031
prob_smarea_error_100corr nulls: 1,430,031 / 1,430,031


In [22]:
print(f"CountyName - distinct values: {ber['CountyName'].nunique()}")
print()
print(ber["CountyName"].value_counts().to_string())

CountyName - distinct values: 55

CountyName
Co. Cork          138261
Co. Dublin        114522
Co. Kildare        68846
Co. Meath          58578
Co. Galway         52463
Co. Wexford        47658
Co. Wicklow        46517
Co. Kerry          43188
Co. Donegal        43165
Co. Tipperary      43048
Co. Louth          41649
Co. Mayo           37506
Co. Clare          33509
Dublin 15          33403
Co. Limerick       31965
Dublin 24          26281
Co. Westmeath      25984
Limerick City      25758
Co. Kilkenny       23470
Co. Laois          22868
Galway City        22017
Dublin 18          21806
Co. Sligo          21024
Co. Waterford      20815
Cork City          19954
Co. Cavan          19949
Dublin 8           19871
Co. Offaly         19486
Dublin 11          17761
Co. Carlow         17601
Dublin 9           17266
Co. Roscommon      17122
Dublin 7           16955
Dublin 12          16755
Waterford City     16566
Dublin 4           14896
Dublin 22          14593
Dublin 16          13936
Co. M

`SA_Code` is entirely null - the column exists in the schema but the values are
stripped from the public extract, so Small Area analysis is not possible.

`CountyName` however contains 55 values rather than 26. Dublin is split into
postal districts, and Cork, Galway, Limerick and Waterford cities are separated
from their surrounding counties. This is more granular than a plain county field
and is the level at which BER data can be joined to the PPR.

### 3.2 Property characteristics

Of the 252 columns, the large majority describe boiler efficiencies, U-values,
solar collector properties and similar technical detail with no bearing on
property value. A small subset is relevant.

In [23]:
keep = [
    "CountyName",
    "DwellingTypeDescr",
    "Year_of_Construction",
    "EnergyRating",
    "BerRating",
    "GroundFloorArea(sq m)",
    "FloorArea",
    "NoStoreys",
    "CO2Rating",
    "MainSpaceHeatingFuel",
]

ber_slim = ber[[col for col in keep if col in ber.columns]].copy()
print(f"Retained {ber_slim.shape[1]} of {ber.shape[1]} columns")
display(ber_slim.head(10))

Retained 10 of 252 columns


,CountyName,DwellingTypeDescr,Year_of_Construction,EnergyRating,BerRating,GroundFloorArea(sq m),FloorArea,NoStoreys,CO2Rating,MainSpaceHeatingFuel
0,Co. Dublin,Detached house,2004,C2,179.59,144.90,90.29,2,38.30,Mains Gas
1,Co. Galway,Detached house,1978,E2,354.13,127.87,127.87,1,87.87,Heating Oil
2,Co. Carlow,Mid-terrace house,2006,C1,165.82,94.78,49.17,2,41.38,Heating Oil
3,Co. Meath,Semi-detached house,2003,C3,200.76,92.89,48.71,2,41.53,Mains Gas
4,Dublin 15,Maisonette,2004,B3,147.50,71.96,0.00,2,27.41,Mains Gas
5,Co. Kildare,Semi-detached house,2018,A2,33.70,138.80,69.40,2,7.27,Electricity
6,Dublin 24,Top-floor apartment,2003,C2,193.69,73.66,0.00,2,36.32,Mains Gas
7,Co. Cork,Detached house,1983,D1,257.72,199.67,164.04,2,56.80,Bulk LPG (propane or butane)
8,Dublin 17,Detached house,1978,C3,209.33,81.98,52.69,2,38.85,Mains Gas
9,Dublin 16,Semi-detached house,2018,A3,50.85,159.50,85.60,2,9.58,Mains Gas


In [24]:
print(ber_slim.describe(include="all").T.to_string())

                           count unique             top    freq         mean         std     min     25%     50%     75%       max
CountyName               1430031     55        Co. Cork  138261          NaN         NaN     NaN     NaN     NaN     NaN       NaN
DwellingTypeDescr        1430031     11  Detached house  438099          NaN         NaN     NaN     NaN     NaN     NaN       NaN
Year_of_Construction   1430031.0    NaN             NaN     NaN  1985.905283   35.386658  1753.0  1974.0  1998.0  2006.0    2104.0
EnergyRating             1430031     21              A2  172431          NaN         NaN     NaN     NaN     NaN     NaN       NaN
BerRating              1430031.0    NaN             NaN     NaN   200.703865  160.597616 -472.99  106.62  179.45  253.59  32134.94
GroundFloorArea(sq m)  1430031.0    NaN             NaN     NaN   118.401559   63.235628    5.47    79.3  103.58  139.24   3546.11
FloorArea              1430031.0    NaN             NaN     NaN    69.853818   49.1

### 3.3 BER data quality issues

The summary statistics reveal several impossible values that will need filtering
in the cleaning stage:

- **`Year_of_Construction`** ranges from 1753 to 2104. Future construction dates
  are clearly data entry errors.
- **`BerRating`** ranges from -472.99 to over 32,000. Energy ratings cannot be
  negative and the upper values are implausible.
- **`CO2Rating`** similarly contains negative values and extreme outliers.
- **`GroundFloorArea(sq m)`** reaches 3,546 sq m, which is not a domestic
  dwelling.
- **`FloorArea`** has a minimum of 0.
- **`NoStoreys`** ranges from 0 to 9.

Note also that `GroundFloorArea(sq m)` and `FloorArea` are different measures
and are not consistent with one another across rows. The schema contains
separate `GroundFloorArea`, `FirstFloorArea`, `SecondFloorArea` and
`ThirdFloorArea` columns, suggesting total dwelling size may need to be
constructed by summing floors rather than taken from a single field.

In [25]:
# quantify how many rows fall outside plausible ranges
checks = {
    "Year_of_Construction < 1700 or > 2026":
        (ber["Year_of_Construction"] < 1700) | (ber["Year_of_Construction"] > 2026),
    "BerRating <= 0 or > 2000":
        (ber["BerRating"] <= 0) | (ber["BerRating"] > 2000),
    "CO2Rating < 0":
        ber["CO2Rating"] < 0,
    "GroundFloorArea <= 10 or > 1000":
        (ber["GroundFloorArea(sq m)"] <= 10) | (ber["GroundFloorArea(sq m)"] > 1000),
    "NoStoreys < 1 or > 5":
        (ber["NoStoreys"] < 1) | (ber["NoStoreys"] > 5),
}

for label, mask in checks.items():
    n = mask.sum()
    print(f"{label:45s} {n:>8,}  ({n / len(ber) * 100:.2f}%)")

Year_of_Construction < 1700 or > 2026                6  (0.00%)
BerRating <= 0 or > 2000                         5,287  (0.37%)
CO2Rating < 0                                    4,744  (0.33%)
GroundFloorArea <= 10 or > 1000                     99  (0.01%)
NoStoreys < 1 or > 5                               104  (0.01%)


All quality issues affect a negligible share of the BER data. The worst is BER rating at 0.37% and CO2 rating at 0.33%, with year of construction, floor area and storey count each below 0.01%. Filtering to plausible ranges removes roughly 10,000 of 1.43 million records and does not materially affect the county-level aggregates the BER data is used for.

In [26]:
print("Dwelling types:")
print(ber["DwellingTypeDescr"].value_counts().to_string())

print("\n\nEnergy ratings:")
print(ber["EnergyRating"].value_counts().to_string())

Dwelling types:
DwellingTypeDescr
Detached house            438099
Semi-detached house       379370
Mid-terrace house         193745
End of terrace house      109026
Mid-floor apartment       108609
Top-floor apartment        77837
Ground-floor apartment     76502
House                      25793
Maisonette                 18407
Apartment                   2069
Basement Dwelling            574


Energy ratings:
EnergyRating
A2    172431
C2    143996
C1    140448
C3    130098
B3    122454
D1    120648
D2    102325
A3     89427
G      74741
B2     69655
E1     59249
F      49383
E2     46527
B1     44161
A1     21200
B      14020
C      10956
A       6624
A0      6614
D       3222
E       1852


## 4. Summary of findings

### Property Price Register

| Item | Finding |
|---|---|
| Rows | 799,067 sales, 2010 to present |
| Price field | Text with mangled euro symbol and comma separators - needs parsing |
| New builds | 140,374 sales listed excluding VAT at 13.5% - needs adjustment |
| Non-market sales | 40,721 flagged, to be excluded from price modelling |
| Property size | 93% null and bands overlap - unusable, will be dropped |
| Eircode coverage | 74-77% from 2022; 242,934 valid records 2021 onward after excluding two placeholder values |
| Routing keys | ~306 distinct, giving useful sub-county granularity |

### SEAI BER

| Item | Finding |
|---|---|
| Rows | 1,430,031 domestic BER assessments |
| Columns | 252, of which roughly 10 are relevant to property value |
| Format | Tab-delimited, latin-1, fixed-width space padding on text fields |
| Small Area code | Entirely null in the public extract - not usable |
| Geography available | County, with Dublin split into postal districts (55 values) |
| Quality | Impossible values in year of construction, BER rating, floor area and storeys |

### Decisions taken

**Two analysis windows.** County-level trend analysis uses the full 2010-present
series of 799k sales. Routing-key-level modelling uses the 2021-present subset
where Eircode coverage is substantial.

**Geographic join is at county and Dublin district level.** Small Area data is
unavailable in the public BER extract and PPR addresses would require geocoding
to reach that granularity. BER characteristics will therefore be joined to PPR
as area-level aggregates - mean floor area, energy rating distribution, dwelling
type mix - rather than per-property.

**The price estimator predicts from location, property type and time**, with
area-level BER characteristics as supporting features. A per-property estimator
using that specific dwelling's floor area and BER rating is not possible from
public data, since the address fields required to join the two datasets at row
level are removed for GDPR compliance.

**Routing key recovery for pre-2021 sales** is worth attempting in notebook 02.
The 243k rows containing both an address and an Eircode form a labelled training
set from which address locality to routing key mappings can be learned, then
applied to the 555k rows lacking an Eircode. Only mappings that are both common
and consistent should be trusted.

### Next

Notebook 02 handles cleaning: price parsing, VAT adjustment, flag filtering,
Irish-language category mapping, BER outlier removal, and the routing key
recovery attempt.